# 🚗 Driver Fatigue Detection - Eye State Classification

This notebook trains a CNN model to classify eye images as **Open** or **Closed** for driver fatigue detection.

**Dataset**: MRL Eyes 2018 (organized into `open_eyes` and `closed_eyes` folders)

### ⚠️ Memory-Safe Design
This notebook uses `tf.data` pipelines to stream images from disk in batches — it will **NOT** crash from MemoryError even on Colab's free tier.

## Step 1: Upload & Setup Dataset

**Option A (Recommended)**: Upload your `train_dataset` folder (with `open_eyes/` and `closed_eyes/` subfolders) as a ZIP to Google Drive, then mount Drive.

**Option B**: Upload the ZIP directly to Colab (slower, resets each session).

In [ ]:
# ============================================================
# OPTION A: Mount Google Drive (recommended)
# ============================================================
# 1. First, zip your train_dataset folder on your PC:
#    Right-click train_dataset -> Send to -> Compressed (zipped) folder
# 2. Upload train_dataset.zip to your Google Drive
# 3. Run this cell to mount Drive

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

In [ ]:
import os
import zipfile

# ============================================================
# UPDATE THIS PATH to where you uploaded the zip on Drive
# ============================================================
ZIP_PATH = '/content/drive/MyDrive/train_dataset.zip'
EXTRACT_TO = '/content/data'

# Create extraction directory
os.makedirs(EXTRACT_TO, exist_ok=True)

# Check if already extracted
dataset_dir = os.path.join(EXTRACT_TO, 'train_dataset')
if os.path.exists(dataset_dir) and os.listdir(dataset_dir):
    print(f"✅ Dataset already extracted at: {dataset_dir}")
else:
    print(f"📦 Extracting {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_TO)
    print(f"✅ Extracted to: {EXTRACT_TO}")

# Verify the structure
for root, dirs, files in os.walk(EXTRACT_TO):
    level = root.replace(EXTRACT_TO, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for d in dirs:
            print(f"{subindent}{d}/")
        if files:
            print(f"{subindent}({len(files)} files)")

## Step 2: Configuration & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import gc

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# ============================================================
# CONFIGURATION - Adjust these as needed
# ============================================================
IMG_SIZE = 80           # Resize images to 80x80 (smaller = less memory)
BATCH_SIZE = 32         # Number of images per batch
EPOCHS = 15             # Training epochs
VALIDATION_SPLIT = 0.2  # 20% for validation
SEED = 42               # Random seed for reproducibility

# ============================================================
# SET YOUR DATASET PATH
# ============================================================
# If your zip contained a 'train_dataset' folder with
# 'open_eyes/' and 'closed_eyes/' subfolders:
DATASET_DIR = '/content/data/train_dataset'

# Verify path exists
assert os.path.exists(DATASET_DIR), f"❌ Dataset not found at {DATASET_DIR}. Check the path!"

# Count images
for class_name in os.listdir(DATASET_DIR):
    class_path = os.path.join(DATASET_DIR, class_name)
    if os.path.isdir(class_path):
        count = len([f for f in os.listdir(class_path) if f.endswith('.png')])
        print(f"  {class_name}: {count} images")

## Step 3: Create Data Pipelines (Memory-Safe)

Using `tf.keras.utils.image_dataset_from_directory` to load images in batches directly from disk — **no MemoryError!**

In [ ]:
# ============================================================
# Create TRAINING dataset (streams from disk, never loads all at once)
# ============================================================
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset='training',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary',    # Binary classification: 0 or 1
    color_mode='rgb',       # 3-channel color
    shuffle=True
)

# ============================================================
# Create VALIDATION dataset
# ============================================================
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary',
    color_mode='rgb',
    shuffle=True
)

# Print class names (alphabetical order: closed_eyes=0, open_eyes=1)
class_names = train_ds.class_names
print(f"\n📋 Classes: {class_names}")
print(f"   0 = {class_names[0]}")
print(f"   1 = {class_names[1]}")

In [ ]:
# ============================================================
# Optimize data pipeline for performance
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

# Cache + Prefetch for faster training
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

print("✅ Data pipeline optimized with prefetching")

## Step 4: Visualize Sample Images

In [ ]:
# ============================================================
# Show sample images from the dataset
# ============================================================
plt.figure(figsize=(12, 6))

for images, labels in train_ds.take(1):
    for i in range(min(16, len(images))):
        ax = plt.subplot(2, 8, i + 1)
        # Images are float [0, 255], normalize to [0, 1] for display
        plt.imshow(images[i].numpy().astype('uint8'))
        label = int(labels[i].numpy())
        plt.title(class_names[label], fontsize=8)
        plt.axis('off')

plt.suptitle('Sample Images from Dataset', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5: Build the CNN Model

In [ ]:
# ============================================================
# Build CNN Model for Eye State Classification
# ============================================================

# Data augmentation layer (applied only during training)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
], name='data_augmentation')

# Model architecture
model = keras.Sequential([
    # Input
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    # Data augmentation
    data_augmentation,

    # Normalize pixel values to [0, 1]
    layers.Rescaling(1.0 / 255),

    # Conv Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Conv Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Conv Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Flatten + Dense layers
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')  # Binary output
], name='eye_state_classifier')

model.summary()

In [ ]:
# ============================================================
# Compile the model
# ============================================================
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Model compiled successfully!")

## Step 6: Train the Model

In [ ]:
# ============================================================
# Callbacks for better training
# ============================================================
callbacks = [
    # Stop if validation loss doesn't improve for 5 epochs
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate when validation loss plateaus
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print("✅ Callbacks configured")
print("🚀 Starting training...")

In [ ]:
# ============================================================
# Train the model
# ============================================================
# Clear memory before training
gc.collect()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")

## Step 7: Evaluate & Visualize Results

In [ ]:
# ============================================================
# Plot training history
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('Model Loss', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Evaluate on validation set
# ============================================================
val_loss, val_accuracy = model.evaluate(val_ds, verbose=1)
print(f"\n📊 Validation Results:")
print(f"   Loss:     {val_loss:.4f}")
print(f"   Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.1f}%)")

In [ ]:
# ============================================================
# Show predictions on sample images
# ============================================================
plt.figure(figsize=(15, 8))

for images, labels in val_ds.take(1):
    predictions = model.predict(images, verbose=0)

    for i in range(min(20, len(images))):
        ax = plt.subplot(4, 5, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))

        pred_label = 1 if predictions[i][0] > 0.5 else 0
        true_label = int(labels[i].numpy())
        confidence = predictions[i][0] if pred_label == 1 else 1 - predictions[i][0]

        color = 'green' if pred_label == true_label else 'red'
        plt.title(
            f"Pred: {class_names[pred_label]}\nTrue: {class_names[true_label]}\nConf: {confidence:.1%}",
            fontsize=7,
            color=color
        )
        plt.axis('off')

plt.suptitle('Model Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 8: Save the Model

In [ ]:
# ============================================================
# Save model to Google Drive (persists across sessions)
# ============================================================
SAVE_PATH = '/content/drive/MyDrive/driver_fatigue_model'

# Save in Keras format
model.save(f'{SAVE_PATH}.keras')
print(f"✅ Model saved to: {SAVE_PATH}.keras")

# Also save as H5 for compatibility
model.save(f'{SAVE_PATH}.h5')
print(f"✅ Model saved to: {SAVE_PATH}.h5")

In [ ]:
# ============================================================
# Load model (for future use)
# ============================================================
# loaded_model = keras.models.load_model('/content/drive/MyDrive/driver_fatigue_model.keras')
# print("✅ Model loaded successfully!")

## Step 9: Test on a Single Image

In [ ]:
# ============================================================
# Test prediction on a single image
# ============================================================
def predict_eye_state(model, image_path, img_size=IMG_SIZE):
    """Predict whether an eye is open or closed."""
    # Load and preprocess
    img = keras.utils.load_img(image_path, target_size=(img_size, img_size))
    img_array = keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Predict
    prediction = model.predict(img_array, verbose=0)[0][0]

    # Interpret
    if prediction > 0.5:
        result = class_names[1]  # open_eyes
        confidence = prediction
    else:
        result = class_names[0]  # closed_eyes
        confidence = 1 - prediction

    # Display
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Prediction: {result}\nConfidence: {confidence:.1%}", fontsize=12)
    plt.axis('off')
    plt.show()

    return result, confidence

# Example usage (uncomment and set path to test):
# result, conf = predict_eye_state(model, '/content/data/train_dataset/open_eyes/some_image.png')
print("✅ Prediction function ready! Uncomment the line above to test.")

---
## 📝 Notes

- **No MemoryError**: This notebook uses `tf.data` pipelines — images are loaded from disk in small batches, never all at once
- **GPU**: Enable GPU in Colab: `Runtime → Change runtime type → GPU`
- **Image Size**: Using 80×80. Increase to 128 or 224 for potentially better accuracy (more memory needed)
- **Batch Size**: If you still get OOM errors on GPU, reduce `BATCH_SIZE` to 16
- **Data**: Make sure your `train_dataset` has two subfolders: `closed_eyes/` and `open_eyes/`